In [ ]:
import reimport stringimport pandas as pdimport numpy as npimport nltkfrom nltk.tokenize import word_tokenizefrom nltk.corpus import stopwordsfrom nltk.sentiment.vader import SentimentIntensityAnalyzerfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import accuracy_scorefrom sklearn.naive_bayes import BernoulliNBfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.tree import DecisionTreeClassifier# Download NLTK resources: tokenizer, stopwords, VADER lexiconnltk.download('punkt')nltk.download('stopwords')nltk.download('vader_lexicon')nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Load scraped reviews data and normalize column namesdf = pd.read_csv('ulasan_aplikasi.csv')# Auto-detect review column ('content' or 'Review') for compatibilitytarget_col = 'content' if 'content' in df.columns else 'Review'# Clean data: remove NaN values, drop duplicates, standardize column name to 'content'clean_df = df[[target_col]].dropna().drop_duplicates()clean_df.columns = ['content']print(f"Total clean reviews for processing: {clean_df.shape[0]}")clean_df.head()

Total ulasan bersih yang akan diproses: 13106


,content
0,The negative reviews on this are hilarious 😭 b...
1,"Good as VN: adequate design encoding, historic..."
2,"epic story, epic gacha"
3,This game is Peak! The illustrations are amazi...
4,"I loved this game, I had fun, I genuinely enjo..."


In [ ]:
def cleaningText(text):    """Remove mentions, hashtags, URLs, numbers, punctuation from text."""    text = re.sub(r'@[A-Za-z0-9]+', '', text)  # Remove @ mentions    text = re.sub(r'#[A-Za-z0-9]+', '', text)  # Remove # hashtags    text = re.sub(r'RT[\s]', '', text)         # Remove RT (retweet)    text = re.sub(r"http\S+", '', text)        # Remove URLs    text = re.sub(r'[0-9]+', '', text)         # Remove digits    text = text.replace('\n', ' ')    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation    text = text.strip(' ')    return textdef casefoldingText(text):    """Convert all characters to lowercase."""    return text.lower()def tokenizingText(text):    """Split text into individual word tokens."""    return word_tokenize(text)def filteringText(text_tokens):    """Remove common English stopwords from token list."""    listStopwords = set(stopwords.words('english'))    return [word for word in text_tokens if word not in listStopwords]def toSentence(list_words):    """Rejoin tokens into a single clean sentence."""    return ' '.join(list_words)# Execute preprocessing pipeline: cleaning → casefolding → tokenizing → stopword removalclean_df['text_clean'] = clean_df['content'].apply(cleaningText)clean_df['text_casefolding'] = clean_df['text_clean'].apply(casefoldingText)clean_df['text_tokenizing'] = clean_df['text_casefolding'].apply(tokenizingText)clean_df['text_stopword'] = clean_df['text_tokenizing'].apply(filteringText)clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)clean_df[['content', 'text_akhir']].head()

,content,text_akhir
0,The negative reviews on this are hilarious 😭 b...,negative reviews hilarious 😭 boo hoo
1,"Good as VN: adequate design encoding, historic...",good vn adequate design encoding historical re...
2,"epic story, epic gacha",epic story epic gacha
3,This game is Peak! The illustrations are amazi...,game peak illustrations amazing stages make st...
4,"I loved this game, I had fun, I genuinely enjo...",loved game fun genuinely enjoyed story god lov...


In [ ]:
# Initialize VADER Sentiment Intensity Analyzer for baseline sentiment scoringsia = SentimentIntensityAnalyzer()def sentiment_analysis_vader(text):    """Analyze sentiment using VADER: compound score ≥ 0 = positive, else negative."""    score = sia.polarity_scores(text)['compound']    polarity = 'positive' if score >= 0 else 'negative'    return score, polarity# Apply VADER sentiment analysis to preprocessed textsresults = clean_df['text_akhir'].apply(sentiment_analysis_vader)results = list(zip(*results))# Add sentiment columns: polarity_score (compound) and polarity (label)clean_df['polarity_score'] = results[0]clean_df['polarity'] = results[1]print("Sentiment Distribution:")print(clean_df['polarity'].value_counts())

Distribusi Sentimen:
polarity
positive    12039
negative     1067
Name: count, dtype: int64


In [ ]:
# Prepare features and target for ML modelsX = clean_df['text_akhir']y = clean_df['polarity']# TF-IDF feature extraction: max 1000 features, min/max doc frequency filterstfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8)X_tfidf = tfidf.fit_transform(X)# Split data: 80% training (model learning), 20% testing (evaluation)X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)print(f"Training data : {X_train.shape[0]} samples")print(f"Test data    : {X_test.shape[0]} samples")

Data Latih: 10484 samples
Data Uji  : 2622 samples


In [ ]:
# Train and evaluate four ML algorithms on TF-IDF featuresml_models = {    "Naive Bayes": BernoulliNB(),    "Logistic Regression": LogisticRegression(),    "Random Forest": RandomForestClassifier(random_state=42),    "Decision Tree": DecisionTreeClassifier(random_state=42)}ml_results = {}print("--- ML Model Evaluation Results (TF-IDF) ---")for name, model in ml_models.items():    # Train model on vectorized training data    model.fit(X_train.toarray(), y_train)    # Predict on test set and calculate accuracy    y_pred_test = model.predict(X_test.toarray())    acc_test = accuracy_score(y_test, y_pred_test)    ml_results[name] = acc_test    print(f"{name:20s} | Test Accuracy: {acc_test:.4f}")

--- Hasil Evaluasi Machine Learning (TF-IDF) ---
Naive Bayes          | Test Accuracy: 0.8978
Logistic Regression  | Test Accuracy: 0.9378
Random Forest        | Test Accuracy: 0.9416
Decision Tree        | Test Accuracy: 0.9100


In [ ]:
# Compare and rank all trained models by accuracysummary_data = {    "Model": ["Naive Bayes", "Logistic Regression", "Random Forest", "Decision Tree"],    "Accuracy": [        ml_results["Naive Bayes"],        ml_results["Logistic Regression"],        ml_results["Random Forest"],        ml_results["Decision Tree"]    ]}# Create summary dataframe and sort by accuracy (best performers first)df_summary = pd.DataFrame(summary_data)df_summary = df_summary.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)print("=== MODEL PERFORMANCE COMPARISON ===")df_summary

=== PERBANDINGAN PERFORMA SEMUA MODEL ===


,Model,Accuracy
0,RoBERTa (Pre-trained),0.955378
1,Random Forest,0.941648
2,Logistic Regression,0.937834
3,Decision Tree,0.909992
4,Naive Bayes,0.897788


In [ ]:
# ========== SENTIMENT PREDICTION WITH RANDOM FOREST ==========# Prerequisite: preprocessing functions (cleaningText, casefoldingText, tokenizingText, filteringText, toSentence)#               TF-IDF vectorizer (tfidf), trained models dict (ml_models)def predict_new_review_rf(text_review):    """    Predict sentiment for a new review using the best-performing Random Forest model.    Returns: (sentiment_label: str, confidence: float)    """    # Step 1: Apply same preprocessing pipeline as training data    cleaned_text = cleaningText(text_review)    cased_text = casefoldingText(cleaned_text)    tokenized_text = tokenizingText(cased_text)    filtered_text = filteringText(tokenized_text)    final_text = toSentence(filtered_text)    # Step 2: Transform text to TF-IDF feature vector    text_vectorized = tfidf.transform([final_text])    # Step 3: Get prediction and confidence scores from Random Forest    rf_model = ml_models["Random Forest"]    predicted_label = rf_model.predict(text_vectorized)[0]    probabilities = rf_model.predict_proba(text_vectorized)[0]    # Step 4: Extract confidence for predicted class    neg_idx = np.where(rf_model.classes_ == 'negative')[0][0]    pos_idx = np.where(rf_model.classes_ == 'positive')[0][0]    if predicted_label == 'positive':        confidence = probabilities[pos_idx]        sentiment_str = "POSITIF"    else:        confidence = probabilities[neg_idx]        sentiment_str = "NEGATIF"    return sentiment_str, confidence# Test inference with sample reviewstest_reviews = [    "The storyline is super amazing and the English voice acting is top-tier!",    "Too many bugs after the update, the game keeps crashing on the loading screen.",    "The game is okay, average gacha mechanics."]print("=== SENTIMENT PREDICTION RESULTS (RANDOM FOREST) ===")for review in test_reviews:    sentimen, conf = predict_new_review_rf(review)    print(f"Review   : \"{review}\"")    print(f"Sentiment: {sentimen} (Confidence: {conf*100:.2f}%)")    print("-" * 50)

=== HASIL INFERENCE SENTIMEN (RANDOM FOREST) ===
Ulasan   : "The storyline is super amazing and the English voice acting is top-tier!"
Sentimen : POSITIF (Confidence: 99.00%)
--------------------------------------------------
Ulasan   : "Too many bugs after the update, the game keeps crashing on the loading screen."
Sentimen : POSITIF (Confidence: 63.00%)
--------------------------------------------------
Ulasan   : "The game is okay, average gacha mechanics."
Sentimen : POSITIF (Confidence: 88.00%)
--------------------------------------------------
